# Data Normalization

This notebook performs normalization on the feature extraction dataset using pre-calculated mean and standard deviation statistics.

## Objectives:
1. Read the comprehensive feature dataset.
2. Read the pre-calculated normalization statistics (mean and std).
3. Verify if the pre-calculated statistics match the actual data.
4. Apply Z-score normalization to numerical features.
5. Save the normalized data to a new CSV file while retaining essential columns like `user_id` and `label`.

In [1]:
import pandas as pd
import numpy as np
import os

# Define paths relative to the notebook location
data_path = r'../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/feature_extraction_V4/feature_extraction_V4_comprehensive_88_users.csv'
stats_path = r'../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/normalization/mean_std_normalization_stats.csv'
output_dir = r'../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/normalization/'
output_file = os.path.join(output_dir, 'normalized_feature_extraction_V4_88_users.csv')

# Load data
print(f"Loading data from {data_path}...")
df = pd.read_csv(data_path)

print(f"Loading stats from {stats_path}...")
stats_df = pd.read_csv(stats_path)

print(f"Data shape: {df.shape}")
print(f"Stats shape: {stats_df.shape}")

Loading data from ../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/feature_extraction_V4/feature_extraction_V4_comprehensive_88_users.csv...
Loading stats from ../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/normalization/mean_std_normalization_stats.csv...
Data shape: (1760, 125)
Stats shape: (111, 3)


In [2]:
# Task 1: Check if the pre-calculated statistics are correct
print("Verifying pre-calculated statistics...")

# Convert stats to a dictionary for easy lookup
stats_dict = stats_df.set_index('Feature').to_dict('index')

verification_results = []
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    if col in stats_dict:
        actual_mean = df[col].mean()
        actual_std = df[col].std()
        
        expected_mean = stats_dict[col]['Mean']
        expected_std = stats_dict[col]['Std']
        
        # Use a small epsilon for floating point comparison
        mean_diff = abs(actual_mean - expected_mean)
        std_diff = abs(actual_std - expected_std)
        
        if mean_diff > 1e-6 or std_diff > 1e-6:
            verification_results.append({
                'Feature': col,
                'Actual Mean': actual_mean,
                'Expected Mean': expected_mean,
                'Mean Diff': mean_diff,
                'Actual Std': actual_std,
                'Expected Std': expected_std,
                'Std Diff': std_diff
            })

if not verification_results:
    print("\nSuccess: All pre-calculated stats match the actual data within tolerance.")
else:
    print(f"\nWarning: Found {len(verification_results)} mismatches in statistics.")
    verify_df = pd.DataFrame(verification_results)
    display(verify_df)

Verifying pre-calculated statistics...

Success: All pre-calculated stats match the actual data within tolerance.


In [3]:
# Task 2: Create normalized dataset
print("Performing normalization...")

# Identify columns to keep
# User specified to keep user_id and label. We also keep any other non-numerical 
# identifier if needed, but here we'll stick to user_id and label as requested.
id_cols = ['user_id', 'label']
numerical_features = [col for col in df.columns if col in stats_dict]

print(f"Keeping identity columns: {id_cols}")
print(f"Normalizing {len(numerical_features)} features.")

# Create final dataframe with selected columns
final_cols = id_cols + numerical_features
normalized_df = df[final_cols].copy()

# Apply Z-score normalization: (x - mean) / std
for col in numerical_features:
    mean = stats_dict[col]['Mean']
    std = stats_dict[col]['Std']
    
    if std > 0:
        normalized_df[col] = (normalized_df[col] - mean) / std
    else:
        # If std is 0, it means all values are the same.
        # We set them to 0.0 to avoid division by zero.
        normalized_df[col] = 0.0

# Task 3: Save the normalized data
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

normalized_df.to_csv(output_file, index=False)
print(f"\nNormalized data saved to: {output_file}")
print(f"Final dataset shape: {normalized_df.shape}")

# Display first few rows
normalized_df.head()

Performing normalization...
Keeping identity columns: ['user_id', 'label']
Normalizing 111 features.

Normalized data saved to: ../../data_processed/feature_kmt_dataset_Edge_Hill_University_22/normalization/normalized_feature_extraction_V4_88_users.csv
Final dataset shape: (1760, 113)


,user_id,label,click_frequency,dwell_gamma_shape,dwell_gamma_loc,dwell_gamma_scale,dwell_lognorm_s,dwell_lognorm_loc,dwell_lognorm_scale,dwell_moment_mean,...,m_angle_gamma_shape,m_angle_gamma_loc,m_angle_gamma_scale,m_angle_lognorm_s,m_angle_lognorm_loc,m_angle_lognorm_scale,m_angle_moment_mean,m_angle_moment_std,m_angle_moment_skew,m_angle_moment_kurtosis
0,1,1,0.608021,-0.079853,0.0,-0.343035,0.061506,0.0,-0.791175,-0.791305,...,-0.027132,0.0,-1.471497,-0.580778,0.0,-2.477796,-3.188438,-2.291365,-0.515334,0.421759
1,1,1,1.097089,-0.055891,0.0,-0.602493,-0.328973,0.0,-0.814131,-0.828779,...,-0.005289,0.0,-2.754482,-3.250164,0.0,0.503947,-1.061537,-3.458488,1.333242,0.062714
2,1,1,0.981948,-0.088347,0.0,-0.198143,0.299119,0.0,-0.763302,-0.755707,...,-0.025903,0.0,-1.259549,-1.167589,0.0,0.761667,0.043017,-0.824803,1.029141,0.148902
3,1,1,0.490358,-0.080049,0.0,-0.324278,0.118126,0.0,-0.739849,-0.740166,...,-0.026200,0.0,-1.150834,-1.345386,0.0,0.747633,0.090890,-0.250905,1.368812,-0.074918
4,1,1,0.406154,-0.094840,0.0,-0.145668,0.422383,0.0,-0.964121,-0.948488,...,-0.021366,0.0,-1.985361,-2.209857,0.0,2.154407,0.965320,-1.676298,-0.086985,0.265218
